# Un transformer actual, pieza a pieza

_Qwen3 en doscientas líneas de PyTorch_

Este cuaderno acompaña a la sección [Qué queda del transformer original](https://iraitzm.github.io/manual-ia-generativa/parts/fundamentos/redes.html) del capítulo sobre redes.

Casi todo lo que se cuenta sobre transformers describe el artículo de 2017. Un modelo de 2026 conserva el esqueleto y ha cambiado cinco piezas, y esas cinco son justo las que explican cuánta memoria consume y a qué velocidad responde. Aquí las montamos una a una y **medimos lo que cada una ahorra**.

No entrenamos nada. Los pesos son aleatorios, así que el modelo no dirá nada con sentido. Lo que interesa son las formas de los tensores y las cuentas de memoria.

> **De dónde sale este código**
>
> La implementación sigue de cerca la de [`LLMs-from-scratch`](https://github.com/rasbt/LLMs-from-scratch) de Sebastian Raschka, que es la referencia habitual para esto. Los nombres de clases y métodos se mantienen en inglés a propósito, para que se puedan cotejar con esa fuente y con el código de Hugging Face.


## Preparación

In [ ]:
%pip install -q torch matplotlib

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(123)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Ejecutando en {device}")

Dos configuraciones. La primera es la real, la de un Qwen3 con mezcla de expertos; la usamos solo para echar cuentas, porque instanciarla necesitaría decenas de gigabytes. La segunda es una maqueta con la misma estructura y todo mucho más pequeño, que sí cabe en cualquier portátil.

In [ ]:
QWEN3_CONFIG = {
    "vocab_size": 151_936,
    "context_length": 262_144,
    "emb_dim": 2048,
    "n_heads": 32,
    "n_layers": 48,
    "head_dim": 128,
    "qk_norm": True,
    "n_kv_groups": 4,
    "rope_base": 1_000_000,
    "dtype": torch.bfloat16,
    "num_experts": 128,
    "num_experts_per_tok": 8,
    "moe_hidden_dim": 768,
}

MAQUETA = {**QWEN3_CONFIG,
    "vocab_size": 2_000,
    "context_length": 512,
    "emb_dim": 128,
    "n_heads": 8,
    "n_layers": 4,
    "head_dim": 16,
    "n_kv_groups": 2,
    "dtype": torch.float32,
    "num_experts": 8,
    "num_experts_per_tok": 2,
    "moe_hidden_dim": 64,
    "hidden_dim": 256,  # para la variante densa
}

## Pieza 1: RMSNorm

Normalizar entre capas evita que los valores se disparen o se apaguen según se avanza en profundidad. El transformer original usaba `LayerNorm`, que resta la media y divide por la desviación típica. Lo que se descubrió después es que **la resta de la media no aportaba casi nada** y sí costaba una pasada más sobre los datos.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6, bias=False, qwen3_compatible=True):
        super().__init__()
        self.eps = eps
        self.qwen3_compatible = qwen3_compatible
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None

    def forward(self, x):
        dtype_entrada = x.dtype

        # Qwen normaliza siempre en float32 aunque el modelo vaya en bfloat16:
        # elevar al cuadrado en precisión baja se desborda con facilidad
        if self.qwen3_compatible:
            x = x.to(torch.float32)

        varianza = x.pow(2).mean(dim=-1, keepdim=True)
        normalizado = x * torch.rsqrt(varianza + self.eps) * self.scale

        if self.shift is not None:
            normalizado = normalizado + self.shift

        return normalizado.to(dtype_entrada)

Comprobemos que hace lo que dice y en qué se diferencia de la versión clásica.

In [ ]:
x = torch.randn(2, 5, 128) * 3 + 7

rms = RMSNorm(128)(x)
ln = nn.LayerNorm(128)(x)

print(f"entrada     media={x.mean():+.3f}  norma cuadrática media={x.pow(2).mean().sqrt():.3f}")
print(f"RMSNorm     media={rms.mean():+.3f}  norma cuadrática media={rms.pow(2).mean().sqrt():.3f}")
print(f"LayerNorm   media={ln.mean():+.3f}  norma cuadrática media={ln.pow(2).mean().sqrt():.3f}")
print(f"\nParámetros: RMSNorm {sum(p.numel() for p in RMSNorm(128).parameters())}, "
      f"LayerNorm {sum(p.numel() for p in nn.LayerNorm(128).parameters())}")

RMSNorm no centra en cero, y resulta que da igual. La mitad de parámetros y una operación menos, multiplicado por dos normalizaciones en cada una de las 48 capas.

## Pieza 2: posición rotatoria

La atención no sabe nada del orden: para ella una frase es un conjunto de tokens. Hay que meterle la posición a mano, y cómo se meta resulta importar mucho más de lo que parecía en 2017, cuando se sumaba un vector fijo a la entrada.

La solución actual es **rotar** los vectores de consulta y clave un ángulo proporcional a su posición, lo que se conoce como [RoPE](https://arxiv.org/abs/2104.09864). La gracia está en una propiedad geométrica: si se rotan dos vectores, el ángulo entre ellos depende solo de **la diferencia** de rotaciones. Es decir, la atención pasa a ver distancias relativas en lugar de posiciones absolutas.

In [ ]:
def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, dtype=torch.float32):
    assert head_dim % 2 == 0, "la dimensión por cabeza tiene que ser par"

    # Cada par de dimensiones gira a una velocidad distinta: las primeras
    # muy rápido (distinguen posiciones cercanas), las últimas muy despacio
    # (distinguen zonas del contexto)
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype) / head_dim))

    posiciones = torch.arange(context_length, dtype=dtype)
    angulos = posiciones.unsqueeze(1) * inv_freq.unsqueeze(0)
    angulos = torch.cat([angulos, angulos], dim=1)

    return torch.cos(angulos), torch.sin(angulos)


def apply_rope(x, cos, sin, offset=0):
    # x: (lote, cabezas, tokens, dimensión por cabeza)
    _, _, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0

    x1 = x[..., : head_dim // 2]
    x2 = x[..., head_dim // 2:]

    cos = cos[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)
    sin = sin[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)

    rotado = torch.cat((-x2, x1), dim=-1)
    return ((x * cos) + (rotado * sin)).to(dtype=x.dtype)

La demostración de que solo cuenta la distancia relativa cabe en cuatro líneas, y es lo que hace que el mecanismo funcione:

In [ ]:
cos, sin = compute_rope_params(head_dim=64, context_length=256)

# El mismo vector colocado en 256 posiciones distintas. Comparamos el producto
# escalar entre pares separados por la misma distancia pero situados lejos.
mismo_vector = torch.randn(64)
w = mismo_vector.expand(1, 1, 256, 64).clone()
wr = apply_rope(w, cos, sin)

print(f"posiciones  10 y  20 → {torch.dot(wr[0,0,10], wr[0,0,20]):.4f}")
print(f"posiciones 110 y 120 → {torch.dot(wr[0,0,110], wr[0,0,120]):.4f}")
print(f"posiciones  10 y  50 → {torch.dot(wr[0,0,10], wr[0,0,50]):.4f}")

Las dos primeras cifras coinciden porque la distancia es la misma. La tercera no, porque la distancia es otra. Nadie programó esa propiedad: sale de rotar.

In [ ]:
plt.figure(figsize=(10, 4))
for dimension in [0, 4, 16, 31]:
    plt.plot(cos[:128, dimension], label=f"dimensión {dimension}")
plt.xlabel("posición en la secuencia")
plt.ylabel("coseno del ángulo")
plt.title("Cada par de dimensiones gira a su ritmo")
plt.legend()
plt.tight_layout()
plt.show()

Ese abanico de frecuencias es lo que permite codificar posiciones muy separadas sin que se confundan. Y `rope_base`, que en Qwen3 vale un millón en lugar de los diez mil originales, es exactamente el mando que estira ese abanico para que aguante contextos largos.

## Pieza 3: atención agrupada

Aquí está el cambio con más consecuencias prácticas. En la atención original cada cabeza tiene su propia clave y su propio valor. En la [agrupada](https://arxiv.org/abs/2305.13245), varias cabezas de consulta **comparten** una clave y un valor.

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_in, num_heads, num_kv_groups, head_dim=None, qk_norm=False, dtype=None):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "las cabezas deben repartirse entre los grupos"

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        if head_dim is None:
            head_dim = d_in // num_heads
        self.head_dim = head_dim
        self.d_out = num_heads * head_dim

        # Aquí está el ahorro: las proyecciones de clave y valor son
        # num_kv_groups veces más pequeñas que la de consulta
        self.W_query = nn.Linear(d_in, self.d_out, bias=False, dtype=dtype)
        self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)
        self.out_proj = nn.Linear(self.d_out, d_in, bias=False, dtype=dtype)

        if qk_norm:
            self.q_norm = RMSNorm(head_dim, eps=1e-6)
            self.k_norm = RMSNorm(head_dim, eps=1e-6)
        else:
            self.q_norm = self.k_norm = None

    def forward(self, x, mask, cos, sin, start_pos=0, cache=None):
        b, num_tokens, _ = x.shape

        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys_new = self.W_key(x).view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values_new = self.W_value(x).view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys_new = self.k_norm(keys_new)

        queries = apply_rope(queries, cos, sin, offset=start_pos)
        keys_new = apply_rope(keys_new, cos, sin, offset=start_pos)

        if cache is not None:
            prev_k, prev_v = cache
            keys = torch.cat([prev_k, keys_new], dim=2)
            values = torch.cat([prev_v, values_new], dim=2)
        else:
            keys, values = keys_new, values_new
        next_cache = (keys, values)

        # Se replican para que cada cabeza de consulta tenga con qué operar.
        # Ojo: esto ocurre en el cálculo, no en la caché, que es lo que importa
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)

        scores = queries @ keys.transpose(2, 3)
        scores = scores.masked_fill(mask, -torch.inf)
        pesos = torch.softmax(scores / self.head_dim**0.5, dim=-1)

        contexto = (pesos @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)
        return self.out_proj(contexto), next_cache

### La cuenta que justifica todo

La **caché KV** guarda las claves y los valores de los tokens ya procesados para no recalcularlos en cada paso de generación. Su tamaño no depende de lo listo que sea el modelo sino de tres números y de la longitud del contexto.

In [ ]:
def tamano_cache_kv(cfg, tokens, agrupada=True):
    """Gigabytes que ocupa la caché KV para un contexto dado."""
    grupos = cfg["n_kv_groups"] if agrupada else cfg["n_heads"]
    bytes_por_valor = torch.tensor([], dtype=cfg["dtype"]).element_size()

    # 2 por clave y valor
    total = 2 * cfg["n_layers"] * grupos * cfg["head_dim"] * tokens * bytes_por_valor
    return total / 1024**3

print(f"{'contexto':>10} {'MHA (2017)':>12} {'GQA (hoy)':>12} {'factor':>8}")
for tokens in [4_096, 32_768, 131_072, 262_144]:
    mha = tamano_cache_kv(QWEN3_CONFIG, tokens, agrupada=False)
    gqa = tamano_cache_kv(QWEN3_CONFIG, tokens, agrupada=True)
    print(f"{tokens:>10,} {mha:>10.1f} GB {gqa:>10.1f} GB {mha/gqa:>7.0f}x")

Esa tabla es, en una línea, la razón por la que hoy se puede servir un contexto de 128.000 tokens sin arruinarse. Y conviene fijarse en algo más: **la caché crece de forma lineal con el contexto y no se comparte entre usuarios**. Multiplicad la columna de la derecha por el número de conversaciones simultáneas y tendréis por qué la memoria de la GPU, y no su potencia de cálculo, es lo que suele limitar cuántas peticiones se atienden a la vez.

### Ejercicio 1

Buscad en la ficha de un modelo abierto reciente sus valores de `num_attention_heads`, `num_key_value_heads`, `num_hidden_layers` y `head_dim`, y calculad su caché KV para 32.000 tokens con la función de arriba.

Después comparadla con el tamaño de sus pesos. En muchos modelos servidos con contextos largos la caché es mayor, y eso sorprende a casi todo el mundo la primera vez.

## Pieza 4: densa o mezcla de expertos

Entre atención y atención hay una red corriente que, según todo indica, es donde se almacena buena parte del conocimiento factual. La versión clásica la aplica entera a cada token.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    def forward(self, x):
        # SwiGLU: una rama decide cuánto deja pasar la otra
        return self.fc3(nn.functional.silu(self.fc1(x)) * self.fc2(x))

La variante con [**mezcla de expertos**](https://arxiv.org/abs/1701.06538) tiene muchas de esas redes y un enrutador que elige unas pocas para cada token. Se paga la memoria de todas y el cálculo de dos o tres.

In [ ]:
class MoEFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.num_experts_per_tok = cfg["num_experts_per_tok"]
        self.emb_dim = cfg["emb_dim"]

        # El enrutador: una capa lineal que puntúa a cada experto
        self.gate = nn.Linear(cfg["emb_dim"], cfg["num_experts"], bias=False, dtype=cfg["dtype"])

        capa = lambda entrada, salida: nn.ModuleList([
            nn.Linear(entrada, salida, bias=False, dtype=cfg["dtype"])
            for _ in range(cfg["num_experts"])
        ])
        self.fc1 = capa(cfg["emb_dim"], cfg["moe_hidden_dim"])
        self.fc2 = capa(cfg["emb_dim"], cfg["moe_hidden_dim"])
        self.fc3 = capa(cfg["moe_hidden_dim"], cfg["emb_dim"])

    def forward(self, x):
        puntuaciones = self.gate(x)
        mejores, indices = torch.topk(puntuaciones, self.num_experts_per_tok, dim=-1)
        probabilidades = torch.softmax(mejores, dim=-1)

        lote, seq_len, _ = x.shape
        x_plano = x.reshape(lote * seq_len, -1)
        salida = torch.zeros_like(x_plano)

        indices_planos = indices.reshape(-1, self.num_experts_per_tok)
        probs_planas = probabilidades.reshape(-1, self.num_experts_per_tok)

        # Se recorre experto a experto y no token a token: cada experto
        # procesa de golpe todos los tokens que le han tocado
        for experto in torch.unique(indices_planos):
            experto = int(experto.item())
            mascara = indices_planos == experto
            seleccion = mascara.any(dim=-1).nonzero(as_tuple=False).squeeze(-1)
            if seleccion.numel() == 0:
                continue

            entrada = x_plano.index_select(0, seleccion)
            oculto = nn.functional.silu(self.fc1[experto](entrada)) * self.fc2[experto](entrada)
            resultado = self.fc3[experto](oculto)

            ranura = mascara[seleccion].int().argmax(dim=-1, keepdim=True)
            peso = torch.gather(probs_planas.index_select(0, seleccion), -1, ranura)

            salida.index_add_(0, seleccion, resultado * peso)

        return salida.reshape(lote, seq_len, self.emb_dim)

Con la configuración real, la diferencia entre parámetros que existen y parámetros que trabajan:

In [ ]:
def parametros_moe(cfg):
    """Parámetros de una capa intermedia con expertos: totales y activos por token."""
    por_experto = 3 * cfg["emb_dim"] * cfg["moe_hidden_dim"]
    enrutador = cfg["emb_dim"] * cfg["num_experts"]

    total = enrutador + cfg["num_experts"] * por_experto
    activo = enrutador + cfg["num_experts_per_tok"] * por_experto
    return total, activo

total, activo = parametros_moe(QWEN3_CONFIG)
print(f"Por capa:  {total/1e6:>8.0f} M parámetros, {activo/1e6:>6.0f} M activos por token")
print(f"48 capas:  {48*total/1e9:>8.1f} B parámetros, {48*activo/1e9:>6.1f} B activos por token")
print(f"\nSe usa el {100*activo/total:.0f}% del modelo para cada token")

Cuando un proveedor anuncia un modelo de decenas de miles de millones de parámetros con solo unos pocos activos, no está haciendo trampa: describe exactamente esto. La consecuencia práctica es incómoda para quien tenga que servirlo, porque **hay que cargar en memoria el modelo entero aunque solo se calcule con una fracción**.

## El bloque y el modelo

Con las cuatro piezas, un bloque es dos subcapas con su normalización y su conexión residual.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"], num_heads=cfg["n_heads"], head_dim=cfg["head_dim"],
            num_kv_groups=cfg["n_kv_groups"], qk_norm=cfg["qk_norm"], dtype=cfg["dtype"],
        )
        self.ff = MoEFeedForward(cfg) if cfg["num_experts"] > 0 else FeedForward(cfg)
        self.norm1 = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.norm2 = RMSNorm(cfg["emb_dim"], eps=1e-6)

    def forward(self, x, mask, cos, sin, start_pos=0, cache=None):
        atajo = x
        x, next_cache = self.att(self.norm1(x), mask, cos, sin, start_pos=start_pos, cache=cache)
        x = x + atajo

        atajo = x
        x = self.ff(self.norm2(x)) + atajo

        return x, next_cache

La normalización va **antes** de cada subcapa y no después, al contrario que en el artículo de 2017. Ese cambio, que parece cosmético, es el que permitió entrenar modelos de decenas de capas sin que la señal se descontrolara.

In [ ]:
class KVCache:
    def __init__(self, n_layers):
        self.cache = [None] * n_layers

    def get(self, layer_idx):
        return self.cache[layer_idx]

    def update(self, layer_idx, value):
        self.cache[layer_idx] = value

    def reset(self):
        self.cache = [None] * len(self.cache)


class Qwen3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])
        self.trf_blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = RMSNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        head_dim = cfg["head_dim"] or cfg["emb_dim"] // cfg["n_heads"]
        cos, sin = compute_rope_params(
            head_dim=head_dim, theta_base=cfg["rope_base"], context_length=cfg["context_length"]
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.current_pos = 0

    def forward(self, in_idx, cache=None):
        x = self.tok_emb(in_idx)
        num_tokens = x.shape[1]

        if cache is not None:
            pos_start = self.current_pos
            pos_end = pos_start + num_tokens
            self.current_pos = pos_end
            mascara = torch.triu(
                torch.ones(pos_end, pos_end, device=x.device, dtype=torch.bool), diagonal=1
            )[pos_start:pos_end, :pos_end]
        else:
            pos_start = 0
            mascara = torch.triu(
                torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool), diagonal=1
            )

        mascara = mascara[None, None, :, :]

        for i, bloque in enumerate(self.trf_blocks):
            x, nueva = bloque(x, mascara, self.cos, self.sin,
                              start_pos=pos_start, cache=cache.get(i) if cache else None)
            if cache is not None:
                cache.update(i, nueva)

        return self.out_head(self.final_norm(x).to(self.cfg["dtype"]))

    def reset_kv_cache(self):
        self.current_pos = 0

Esa máscara triangular es lo que convierte al modelo en generativo: impide que un token mire a los que vienen después. Merece la pena verla.

In [ ]:
n = 8
plt.figure(figsize=(4, 4))
plt.imshow(torch.triu(torch.ones(n, n), diagonal=1), cmap="Greys")
plt.title("En negro, lo que cada token\nno puede mirar")
plt.xlabel("token mirado")
plt.ylabel("token que mira")
plt.tight_layout()
plt.show()

## Que funcione

In [ ]:
modelo = Qwen3Model(MAQUETA).to(device)

entrada = torch.randint(0, MAQUETA["vocab_size"], (1, 12), device=device)
with torch.no_grad():
    logits = modelo(entrada)

print(f"entrada: {tuple(entrada.shape)}  (lote, tokens)")
print(f"salida:  {tuple(logits.shape)}  (lote, tokens, vocabulario)")
print(f"\nParámetros de la maqueta: {sum(p.numel() for p in modelo.parameters()):,}")

Una distribución de probabilidad sobre el vocabulario **para cada posición**. De ahí sale el token siguiente, y repitiendo el proceso sale la respuesta entera.

### Con caché y sin caché

Aquí se ve para qué sirve la caché KV, que es la quinta pieza y la única que no está en los pesos sino en cómo se ejecutan.

In [ ]:
import time

def generar(modelo, prompt, nuevos_tokens, usar_cache):
    modelo.reset_kv_cache()
    cache = KVCache(modelo.cfg["n_layers"]) if usar_cache else None
    secuencia = prompt

    with torch.no_grad():
        if usar_cache:
            logits = modelo(prompt, cache=cache)
            for _ in range(nuevos_tokens):
                siguiente = logits[:, -1].argmax(dim=-1, keepdim=True)
                secuencia = torch.cat([secuencia, siguiente], dim=1)
                logits = modelo(siguiente, cache=cache)
        else:
            for _ in range(nuevos_tokens):
                logits = modelo(secuencia)          # recalcula todo cada vez
                siguiente = logits[:, -1].argmax(dim=-1, keepdim=True)
                secuencia = torch.cat([secuencia, siguiente], dim=1)

    return secuencia


prompt = torch.randint(0, MAQUETA["vocab_size"], (1, 64), device=device)

for usar_cache in [False, True]:
    inicio = time.perf_counter()
    salida = generar(modelo, prompt, nuevos_tokens=48, usar_cache=usar_cache)
    print(f"caché={str(usar_cache):5} → {time.perf_counter() - inicio:.2f} s, "
          f"{salida.shape[1]} tokens")

Sin caché, cada token nuevo obliga a reprocesar la secuencia entera. Con caché, solo se procesa el token nuevo y los anteriores se recuperan de memoria. La diferencia crece con la longitud, que es la razón de que sea obligatoria en cualquier sistema real.

### Ejercicio 2

Repetid la medida con prompts de 16, 64, 256 y 512 tokens y dibujad las dos curvas.

Sin caché el tiempo debería crecer de forma claramente superlineal; con caché, casi recta. Esa diferencia de forma, y no el valor concreto, es lo que hay que llevarse.

## Las cuentas del modelo real

Ya se puede calcular lo que ocupa el modelo de verdad sin instanciarlo.

In [ ]:
def parametros_totales(cfg):
    e, h, hd = cfg["emb_dim"], cfg["n_heads"], cfg["head_dim"]

    embeddings = cfg["vocab_size"] * e
    atencion = e * h * hd + 2 * (e * cfg["n_kv_groups"] * hd) + h * hd * e
    if cfg["qk_norm"]:
        atencion += 2 * hd
    intermedia, activa = parametros_moe(cfg)
    normalizaciones = 2 * e

    por_capa = atencion + intermedia + normalizaciones
    activos_por_capa = atencion + activa + normalizaciones

    total = embeddings + cfg["n_layers"] * por_capa + e + cfg["vocab_size"] * e
    activos = embeddings + cfg["n_layers"] * activos_por_capa + e + cfg["vocab_size"] * e
    return total, activos


total, activos = parametros_totales(QWEN3_CONFIG)
bytes_por_valor = torch.tensor([], dtype=QWEN3_CONFIG["dtype"]).element_size()

print(f"Parámetros totales:  {total/1e9:6.1f} B")
print(f"Activos por token:   {activos/1e9:6.1f} B")
print(f"\nPesos en bfloat16:   {total * bytes_por_valor / 1024**3:6.1f} GB")
print(f"Pesos en float32:    {total * 4 / 1024**3:6.1f} GB")
print(f"Caché a 32k tokens:  {tamano_cache_kv(QWEN3_CONFIG, 32_768):6.1f} GB")

Esas tres últimas líneas son la respuesta a "¿me cabe este modelo en mi GPU?", que es la pregunta que se hace todo el mundo antes de descargar nada. Y explican de paso por qué la cuantización, que baja los bytes por valor, es la primera palanca que se toca.

### Ejercicio 3

Añadid una cuarta línea con la memoria que haría falta para **entrenar** el modelo, no solo para ejecutarlo.

Pista: además de los pesos hacen falta los gradientes, del mismo tamaño, y el estado del optimizador, que con Adam son dos valores más por parámetro y normalmente en float32. El resultado explica por qué el ajuste fino completo es tan caro y por qué existen técnicas como LoRA.

## Para seguir

* [Inferencia](https://iraitzm.github.io/manual-ia-generativa/parts/modelos/inferencia.html), donde estas cuentas se convierten en latencia y en factura.
* [Panorama de modelos](https://iraitzm.github.io/manual-ia-generativa/parts/modelos/panorama.html), para poner la configuración de arriba en contexto.
* [`LLMs-from-scratch`](https://github.com/rasbt/LLMs-from-scratch), si queréis la versión larga con entrenamiento incluido.